# CueX Phase 3 — Colab 双服务共享 GPU 部署与验证（ASR + JEV-27B System 1）

本 Notebook 在 **Google Colab G4 (`NVIDIA RTX PRO 6000 Blackwell Server Edition`)** 上验证两项同卡 GPU 推理服务（严禁部署 System 2 大模型）：

- **端口 `10086`（GPU ASR）**：`FunAudioLLM/SenseVoiceSmall` + `funasr/fsmn-vad`（HTTPS + WSS 实时流式语音识别，零落盘音频）。
- **端口 `10010`（Jev-Like System 1）**：`autotrust/JEV-27B`（Qwen3.8-27B BF16 + LoRA + `head.safetensors` 24 槽校准头，HTTPS + Bearer 鉴权，兼容 `llama.cpp` `/completion` 与 `/apply-template`）。

> **安全铁律**：本 Notebook 与输出单元绝不保存或打印任何 API Key、FRP `auth.token`、SSH 私钥、公网 IP 或个人主目录路径。


In [1]:
import os
import platform
import subprocess
import sys
import torch

print("Python:", sys.version.split()[0], "| Arch:", platform.machine())
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "| CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "CUDA GPU 运行时不可用"
props = torch.cuda.get_device_properties(0)
free_b, total_b = torch.cuda.mem_get_info(0)
print(
    f"GPU: {props.name} | CC: ({props.major}, {props.minor}) | "
    f"总显存: {total_b / (1024**2):.1f} MiB | 当前空闲显存: {free_b / (1024**2):.1f} MiB"
)


Python: 3.13.16 | Arch: x86_64
PyTorch: 2.14.1+cu130 | CUDA available: True | CUDA: 13.0
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition | CC: (12, 0) | 总显存: 97250.1 MiB | 当前空闲显存: 41377.9 MiB


In [2]:
import importlib.metadata as md

# 校验 ASR 与 JEV-27B System 1 所需的核心依赖包版本
required_pkgs = [
    "funasr",
    "modelscope",
    "transformers",
    "peft",
    "accelerate",
    "safetensors",
    "huggingface-hub",
    "fastapi",
    "uvicorn",
    "websockets",
    "cryptography",
    "python-multipart",
    "numpy",
]
for pkg in required_pkgs:
    print(f"{pkg:18s}: {md.version(pkg)}")


funasr            : 1.4.26
modelscope        : 1.40.3
transformers      : 5.22.1
peft              : 0.21.2
accelerate        : 1.15.0
safetensors       : 0.8.0
huggingface-hub   : 1.33.0
fastapi           : 0.141.1
uvicorn           : 0.54.0
websockets        : 15.0.1
cryptography      : 50.0.2
python-multipart  : 0.0.33
numpy             : 2.4.6


In [3]:
import shutil
import subprocess

# 仅校验 frpc 与 sshd 进程与版本状态，绝不读取或打印 frpc.toml 密钥内容
frpc_bin = "/root/frp-ssh/frpc" if os.path.exists("/root/frp-ssh/frpc") else shutil.which("frpc")
if frpc_bin:
    ver = subprocess.check_output([frpc_bin, "--version"], text=True).strip()
    running = subprocess.run(["pgrep", "-x", "frpc"], capture_output=True, text=True).returncode == 0
    print(f"frpc binary: present (version={ver}) | active_daemon={running}")
else:
    print("frpc binary: not installed (loopback-only mode)")

sshd_active = subprocess.run(["pgrep", "-x", "sshd"], capture_output=True, text=True).returncode == 0
print(f"sshd daemon: active={sshd_active}")


frpc binary: present (version=0.71.1) | active_daemon=True
sshd daemon: active=True


In [4]:
import time
import wave
from pathlib import Path
import numpy as np
from funasr import AutoModel
from funasr.utils.postprocess_utils import rich_transcription_postprocess

t_load0 = time.perf_counter()
asr_model = AutoModel(
    model="FunAudioLLM/SenseVoiceSmall",
    vad_model="fsmn-vad",
    vad_kwargs={"max_single_segment_time": 30000},
    device="cuda:0",
    hub="hf",
    disable_update=True,
)
load_sec = time.perf_counter() - t_load0

# 内存静音预热
_ = asr_model.generate(input=np.zeros(16000, dtype=np.float32), cache={}, language="auto", use_itn=True, disable_pbar=True)

sample_wav = Path("/content/cuex-benchmark-audio/01_intro_natural_tingting.wav")
if sample_wav.is_file():
    with wave.open(str(sample_wav), "rb") as wf:
        pcm = np.frombuffer(wf.readframes(wf.getnframes()), dtype=np.int16).astype(np.float32) / 32768.0
        dur_sec = len(pcm) / wf.getframerate()
    source_desc = sample_wav.name
else:
    dur_sec = 1.5
    pcm = (0.05 * np.sin(2.0 * np.pi * 220.0 * np.arange(int(16000 * dur_sec)) / 16000.0)).astype(np.float32)
    source_desc = "synthetic-1.5s-sine"

t_inf0 = time.perf_counter()
out = asr_model.generate(input=pcm, cache={}, language="auto", use_itn=True, merge_vad=True, disable_pbar=True)
inf_ms = (time.perf_counter() - t_inf0) * 1000.0
text = rich_transcription_postprocess(out[0]["text"]).strip() if out else ""
rtf = (inf_ms / 1000.0) / max(dur_sec, 1e-3)
print(f"ASR 加载耗时: {load_sec:.3f}s | 样本: {source_desc} ({dur_sec:.2f}s)")
print(f"GPU 推理: {inf_ms:.2f} ms | RTF: {rtf:.5f}")
print(f"转写预览: {text[:80]}")

del asr_model
torch.cuda.empty_cache()


funasr version: 1.4.26.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 29 files:   0%|          | 0/29 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

... (4 outputs left out)

{'load_data': '0.000', 'extract_feat': '0.003', 'forward': '0.026', 'batch_size': '1', 'rtf': '0.002'}, : 100%|██████████| 1/1 [00:00<00:00, 37.44it/s]

rtf_avg: 0.002: 100%|██████████| 1/1 [00:00<00:00, 36.83it/s]                                                                                          

rtf_avg: 0.002: 100%|██████████| 1/1 [00:00<00:00, 36.02it/s]

ASR 加载耗时: 11.310s | 样本: 01_intro_natural_tingting.wav (16.26s)
GPU 推理: 128.22 ms | RTF: 0.00788
转写预览: 各位评委好，我们是进行好了一点团队，带来的产品叫gueXQX是在深度思考之外，接触当下的快思考外挂，帮助演讲者在台上保持眼神交流与核心表达。


In [5]:
import asyncio
import json
import os
import secrets
import shutil
import socket
import ssl
import subprocess
import sys
import tempfile
import time
import urllib.error
import urllib.request
from pathlib import Path
import websockets

# 在临时空闲回环端口启动独立 ASR 测试子进程，不干扰线上 10086 与 10010 服务
server_script = Path("/opt/cuex-asr/start_asr.py")
assert server_script.is_file(), f"Missing {server_script}"

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.bind(("127.0.0.1", 0))
    test_port = s.getsockname()[1]

ephemeral_token = secrets.token_hex(24)  # 绝不打印
tmp_tls_dir = tempfile.mkdtemp(prefix="cuex-asr-nb-tls-")
env = os.environ.copy()
env.update({
    "COLAB_ASR_HOST": "127.0.0.1",
    "COLAB_ASR_PORT": str(test_port),
    "COLAB_ASR_MODEL": "FunAudioLLM/SenseVoiceSmall",
    "COLAB_ASR_HUB": "hf",
    "COLAB_ASR_DEVICE": "cuda:0",
    "COLAB_ASR_API_KEY": ephemeral_token,
    "COLAB_ASR_TLS_DIR": tmp_tls_dir,
    "COLAB_ASR_TLS_CERT": f"{tmp_tls_dir}/cert.pem",
    "COLAB_ASR_TLS_KEY": f"{tmp_tls_dir}/key.pem",
    "COLAB_ASR_TLS_SAN_IPS": "127.0.0.1",
    "COLAB_ASR_TLS_SAN_DNS": "localhost",
})

proc = subprocess.Popen([sys.executable, str(server_script)], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    base_url = f"https://127.0.0.1:{test_port}"
    cert_path = Path(tmp_tls_dir) / "cert.pem"
    ready_payload = None

    for _ in range(45):
        if cert_path.is_file():
            ssl_ctx = ssl.create_default_context(cafile=str(cert_path))
            req = urllib.request.Request(f"{base_url}/ready", headers={"Authorization": f"Bearer {ephemeral_token}"})
            try:
                with urllib.request.urlopen(req, context=ssl_ctx, timeout=2.0) as r:
                    if r.status == 200:
                        ready_payload = json.loads(r.read().decode("utf-8"))
                        break
            except Exception:
                pass
        time.sleep(1.0)

    assert ready_payload and ready_payload.get("ready") is True, "Notebook loopback ASR server failed to become ready"
    ssl_ctx = ssl.create_default_context(cafile=str(cert_path))

    # 1. 验证未带 Token 返回 401
    unauth_code = None
    try:
        urllib.request.urlopen(f"{base_url}/health", context=ssl_ctx, timeout=3.0)
    except urllib.error.HTTPError as e:
        unauth_code = e.code
    assert unauth_code == 401, f"Expected 401, got {unauth_code}"

    # 2. 内存音频批量转写验证
    test_wav = Path("/content/cuex-benchmark-audio/01_intro_natural_tingting.wav")
    if test_wav.is_file():
        payload_bytes = test_wav.read_bytes()
        ctype = "audio/wav"
    else:
        payload_bytes = (np.zeros(16000, dtype=np.int16)).tobytes()
        ctype = "audio/pcm"

    tx_req = urllib.request.Request(
        f"{base_url}/v1/asr/transcribe",
        data=payload_bytes,
        headers={"Authorization": f"Bearer {ephemeral_token}", "Content-Type": ctype},
        method="POST",
    )
    with urllib.request.urlopen(tx_req, context=ssl_ctx, timeout=15.0) as tx_resp:
        tx_json = json.loads(tx_resp.read().decode("utf-8"))
    assert tx_json.get("ok") is True, f"Batch transcribe failed: {tx_json}"

    # 3. 鉴权 WSS 流式分片验证
    async def ws_smoke() -> dict:
        ws_url = f"wss://127.0.0.1:{test_port}/v1/asr/stream"
        async with websockets.connect(ws_url, ssl=ssl_ctx, additional_headers={"Authorization": f"Bearer {ephemeral_token}"}) as ws:
            ready_msg = json.loads(await ws.recv())
            assert ready_msg["type"] == "ready"
            pcm_body = payload_bytes[44:] if ctype == "audio/wav" else payload_bytes
            chunk_sz = 32 * 100
            for i in range(0, min(len(pcm_body), 32 * 4000), chunk_sz):
                await ws.send(pcm_body[i:i + chunk_sz])
                await asyncio.sleep(0.01)
            await ws.send("Done")
            events = []
            async for msg in ws:
                ev = json.loads(msg)
                events.append(ev)
                if ev.get("type") == "done":
                    break
            return {"ready": True, "eventTypes": [e.get("type") for e in events]}

    ws_res = await ws_smoke()
    print(json.dumps({
        "asrLoopbackSmokePassed": True,
        "unauthStatus": unauth_code,
        "batchOk": tx_json["ok"],
        "batchLatencyMs": tx_json["latencyMs"],
        "batchRtf": tx_json["rtf"],
        "wsEventTypes": ws_res["eventTypes"],
    }, ensure_ascii=False, indent=2))
finally:
    proc.terminate()
    try:
        proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proc.kill()
    shutil.rmtree(tmp_tls_dir, ignore_errors=True)


{
  "asrLoopbackSmokePassed": true,
  "unauthStatus": 401,
  "batchOk": true,
  "batchLatencyMs": 186.34,
  "batchRtf": 0.01146,
  "wsEventTypes": [
    "final",
    "done"
  ]
}


In [6]:
import json
import math
import ssl
import urllib.error
import urllib.request
from pathlib import Path
from safetensors import safe_open

# 1. 校验本地下载的 autotrust/JEV-27B 校准与权重产物完整性（不重复加载 54GB 权重以免与在线 :10010 冲突）
model_root = Path("/opt/cuex-jev/model")
for rel in ["config.json", "calibration.json", "judge_config.json", "head.safetensors", "adapter/adapter_model.safetensors", "adapter_vllm/decision_head.json"]:
    assert (model_root / rel).is_file(), f"缺少 JEV-27B 文件: {model_root / rel}"

calib = json.loads((model_root / "calibration.json").read_text())
dec_meta = json.loads((model_root / "adapter_vllm/decision_head.json").read_text())
with safe_open(str(model_root / "head.safetensors"), framework="pt") as sf:
    w_shape = tuple(sf.get_tensor("proj.weight").shape)
    b_tensor = sf.get_tensor("proj.bias")
    max_bias_diff = float((b_tensor - torch.tensor(dec_meta["bias"], dtype=b_tensor.dtype)).abs().max().item())

print(json.dumps({
    "jevModelDir": str(model_root),
    "headWeightShape": w_shape,
    "headBiasDiffVsJson": max_bias_diff,
    "perKindTemperatures": calib["per_kind"],
}, ensure_ascii=False, indent=2))
assert max_bias_diff < 1e-6

# 2. 内存读取本地权限受限环境文件，对运行中的 127.0.0.1:10010 (JEV-27B) 与 127.0.0.1:10086 (ASR) 执行同卡在线验收
def _read_kv(p: Path) -> dict[str, str]:
    out = {}
    for line in p.read_text().splitlines():
        s = line.strip()
        if s and not s.startswith("#") and "=" in s:
            k, v = s.split("=", 1)
            out[k.strip()] = v.strip()
    return out

s1_env = _read_kv(Path("/opt/cuex-jev/.env.s1"))
asr_env = _read_kv(Path("/opt/cuex-asr/.env.colab"))
s1_key = s1_env["COLAB_S1_API_KEY"]  # 仅存内存，绝不打印
asr_key = asr_env["COLAB_ASR_API_KEY"]
ctx = ssl.create_default_context(cafile="/tmp/cuex-asr-tls/cert.pem")

# 2a. 验证 S1 端口 10010 拒绝未鉴权请求 (401)
unauth_s1 = None
try:
    urllib.request.urlopen("https://127.0.0.1:10010/health", context=ctx, timeout=3.0)
except urllib.error.HTTPError as err:
    unauth_s1 = err.code
assert unauth_s1 == 401

# 2b. 验证 S1 /health、/completion (unbiased W@h) 与 /v1/decide (PyTorch 直接前向 W@h+b) 数值一致性
with urllib.request.urlopen(urllib.request.Request("https://127.0.0.1:10010/health", headers={"Authorization": f"Bearer {s1_key}"}), context=ctx, timeout=5.0) as r:
    s1_health = json.loads(r.read().decode("utf-8"))

with urllib.request.urlopen(urllib.request.Request("https://127.0.0.1:10086/health", headers={"Authorization": f"Bearer {asr_key}"}), context=ctx, timeout=5.0) as r:
    asr_health = json.loads(r.read().decode("utf-8"))

state = '{"goal":"回答评委提问","chapter":"ch_qa","silenceMs":200}'
question = "Does speaker need grounded Q&A help?"
prompt = f"[kind] noul\n[state] {state}\n[question] {question}\n[options]\nfalse\ntrue\n[decision]:"
vids = dec_meta["verbalizer_ids"][0:2]
bias2 = dec_meta["bias"][0:2]
temp_noul = float(calib["per_kind"]["noul"])

req_comp = urllib.request.Request(
    "https://127.0.0.1:10010/completion",
    data=json.dumps({
        "prompt": prompt,
        "n_predict": 1,
        "grammar": 'root ::= "false" | "true"',
        "logit_bias": [[vids[0], 100], [vids[1], 100]],
        "n_probs": 32,
        "post_sampling_probs": True,
        "temperature": 1,
        "lora": [{"id": 0, "scale": 1}],
    }).encode("utf-8"),
    headers={"Authorization": f"Bearer {s1_key}", "Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(req_comp, context=ctx, timeout=10.0) as r:
    comp = json.loads(r.read().decode("utf-8"))

req_dir = urllib.request.Request(
    "https://127.0.0.1:10010/v1/decide",
    data=json.dumps({"kind": "noul", "state": state, "question": question}).encode("utf-8"),
    headers={"Authorization": f"Bearer {s1_key}", "Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(req_dir, context=ctx, timeout=10.0) as r:
    direct = json.loads(r.read().decode("utf-8"))

entries = {int(e["id"]): float(e["prob"]) for e in comp["completion_probabilities"][0]["top_probs"]}
lw = [(math.log(entries[vids[i]]) + bias2[i]) / temp_noul for i in range(2)]
mx = max(lw)
ex = [math.exp(x - mx) for x in lw]
p_compat = [v / sum(ex) for v in ex]
p_direct = [float(direct["probabilities"]["false"]), float(direct["probabilities"]["true"])]
max_prob_diff = max(abs(a - b) for a, b in zip(p_compat, p_direct))
assert max_prob_diff < 1e-4

print(json.dumps({
    "colocatedVerificationPassed": True,
    "s1UnauthStatus": unauth_s1,
    "s1Model": s1_health["model"],
    "s1Calibrated": s1_health["calibrated"],
    "s1ServerLatencyMs": comp["latencyMs"],
    "maxProbDiffVsPyTorchDirect": round(max_prob_diff, 8),
    "asrModel": asr_health["model"],
    "asrReady": asr_health["ready"],
    "gpuDeviceUsedMiB": s1_health["gpu"]["vramDeviceUsedMiB"],
    "gpuFreeMiB": s1_health["gpu"]["vramFreeMiB"],
    "gpuTotalMiB": s1_health["gpu"]["vramTotalMiB"],
}, ensure_ascii=False, indent=2))


{
  "jevModelDir": "/opt/cuex-jev/model",
  "headWeightShape": [
    24,
    5120
  ],
  "headBiasDiffVsJson": 0.0,
  "perKindTemperatures": {
    "noul": 1.0143134751376188,
    "choice": 1.0161280671450397,
    "score": 1.0036213515883572
  }
}
{
  "colocatedVerificationPassed": true,
  "s1UnauthStatus": 401,
  "s1Model": "autotrust/JEV-27B",
  "s1Calibrated": true,
  "s1ServerLatencyMs": 81.51,
  "maxProbDiffVsPyTorchDirect": 5e-08,
  "asrModel": "FunAudioLLM/SenseVoiceSmall",
  "asrReady": true,
  "gpuDeviceUsedMiB": 56006.2,
  "gpuFreeMiB": 41243.8,
  "gpuTotalMiB": 97250.1
}


## 生产环境启动对照（Port `10086` ASR + Port `10010` JEV-27B）

1. 将随机生成的 `COLAB_ASR_API_KEY` 与 `COLAB_S1_API_KEY` 仅写入 Colab 权限受限文件（`chmod 600`）与本地 gitignored `.cuex-runtime/.env.phase3.local`（同步保存 `CUEX_S1_*` 与 `CUEX_COLAB_ASR_*`）。
2. 后台同时运行 `start_asr.py`（`127.0.0.1:10086`）与 `start_jev_s1.py`（`127.0.0.1:10010`），经 `frpc 0.71.1` 映射到公网端口。
3. 本机仅复制公钥 `/tmp/cuex-asr-tls/cert.pem` 至 `.cuex-runtime/colab-asr-cert.pem`，通过 `infra/colab/healthcheck.py` 与 `infra/colab/healthcheck_jev.py` 完成静默全量验收。
